# Section 1 & 1.5: Validation via Self-Induced Confounding

This notebook proves the estimation pipeline works before trusting it on anything else.

**Sequence:**
1. Load the Criteo Uplift v2.1 dataset and validate its integrity.
2. Compute the ground-truth ATE from the full randomized data (`visit`, with `conversion` reported alongside).
3. Section 1.5: run the MDE calculation that justifies using `visit`, not `conversion`, for the rest of the project.
4. Select a confounding covariate by correlation with the outcome (not arbitrarily).
5. Calibrate a retention rule that induces real confounding, validated against a two-part gate.
6. Run the dose-response comparison (naive OLS / PSM / IPW / AIPW) across confounding severities.
7. Log every run, save balance diagnostics, and save the matched-pairs sample for Section 4.

All heavy lifting lives in `src/`; this notebook only calls into it and records results.

In [1]:
import os, sys

# Jupyter sets the kernel's cwd to this notebook's own folder (notebooks/),
# but src/ lives one level up, as a sibling of notebooks/, not a child of it.
# This makes 'from src...' imports resolve regardless of how the notebook
# was launched (jupyter notebook from the project root vs from notebooks/).
_cwd = os.getcwd()
_project_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == "notebooks" else _cwd
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)


In [2]:
import json
import os

import numpy as np
import pandas as pd

from src.utils.logging_config import configure_logging
from src.utils.data_loader import load_criteo_uplift
from src.utils.power_analysis import mde_comparison_table
from src.utils.bootstrap import analytic_ci_diff_in_proportions
from src.validation.confounding import select_confounding_covariate, calibrate_confounding, run_dose_response_confounding
from src.validation.estimators import run_estimator_comparison, fit_propensity_score
from src.validation.diagnostics import run_full_diagnostics
from src.utils.db import log_estimation_run

configure_logging()

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/interim", exist_ok=True)

RANDOM_STATE = 42
SUBSAMPLE_N = 300_000  # CPU-feasible subsample for confounding/estimator work; ground-truth ATE uses full data

14:33:21.398 Structured logging configured via Logfire


## 1. Load and validate the dataset

In [3]:
df = load_criteo_uplift(validate=True)
print(df.shape)
df.head()

14:33:21.408 Attempting load from Hugging Face (criteo/criteo-uplift)


14:33:23.103 HTTP Request: HEAD https://huggingface.co/datasets/criteo/criteo-uplift/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
14:33:23.103 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
14:33:23.261 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/criteo/criteo-uplift/2424920019e49d52d72c13ac1143ec5d53af276b/README.md "HTTP/1.1 200 OK"
14:33:23.724 HTTP Request: HEAD https://huggingface.co/datasets/criteo/criteo-uplift/resolve/2424920019e49d52d72c13ac1143ec5d53af276b/criteo-uplift.py "HTTP/1.1 404 Not Found"


Logfire project URL: https://logfire-us.pydantic.dev/abhinewnpc/ticket-triage-mlops

14:33:25.062 HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/criteo/criteo-uplift/criteo/criteo-uplift.py "HTTP/1.1 404 Not Found"
14:33:25.326 HTTP Request: GET https://huggingface.co/api/datasets/criteo/criteo-uplift/revision/2424920019e49d52d72c13ac1143ec5d53af276b "HTTP/1.1 200 OK"
14:33:25.609 HTTP Request: HEAD https://huggingface.co/datasets/criteo/criteo-uplift/resolve/2424920019e49d52d72c13ac1143ec5d53af276b/.huggingface.yaml "HTTP/1.1 404 Not Found"
14:33:27.187 HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=criteo/criteo-uplift "HTTP/1.1 200 OK"
14:33:27.454 HTTP Request: GET https://huggingface.co/api/datasets/criteo/criteo-uplift/tree/2424920019e49d52d72c13ac1143ec5d53af276b/data?recursive=true&expand=false "HTTP/1.1 404 Not Found"
14:33:27.743 HTTP Request: GET https://huggingface.co/api/datasets/criteo/criteo-uplift/tree/2424920019e49d52d72c13ac1143ec5d53af276b?recursive=false&expand=false "HTTP/1.1 200 OK"
14:

criteo-research-uplift-v2.1.csv.gz: reconstructing file:   0%|          |  0.00B /  311MB            

criteo-research-uplift-v2.1.csv.gz: downloading bytes:           |  0.00B            

c:\Users\abhin\Desktop\Important\codes\projects\causal-impact-lab\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abhin\.cache\huggingface\hub\datasets--criteo--criteo-uplift. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Generating train split:   0%|          | 0/13979592 [00:00<?, ? examples/s]

14:37:21.181 Hugging Face load succeeded
14:37:21.181 Row count check passed: 13979592 rows
14:37:21.198 Treatment split check passed: 0.8500 treated
14:37:21.221 Outcome rate check passed: visit=0.0470, conversion=0.0029
(13979592, 16)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [4]:
FEATURE_COLS = [f"f{i}" for i in range(12)]
print(FEATURE_COLS)

['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']


## 2. Ground-truth ATE (full randomized dataset)

At this sample size (~13.9M rows) the analytic Wald CI is already essentially exact;
bootstrapping here would cost significant compute for no statistical gain (see
`utils/bootstrap.py` docstring). `visit` is the primary outcome; `conversion` is reported
alongside as a secondary full-dataset ATE only, per the Section 1.5 justification below.

In [5]:
def compute_ground_truth(df, outcome_col, treatment_col):
    treated = df[df[treatment_col] == 1][outcome_col]
    control = df[df[treatment_col] == 0][outcome_col]
    return analytic_ci_diff_in_proportions(
        treated.mean(), len(treated), control.mean(), len(control)
    )

ground_truth_visit = compute_ground_truth(df, "visit", "treatment")
ground_truth_conversion = compute_ground_truth(df, "conversion", "treatment")

print("visit ATE:", ground_truth_visit)
print("conversion ATE (secondary, full-dataset only):", ground_truth_conversion)

with open("../data/processed/ground_truth.json", "w") as f:
    json.dump(
        {
            "ate": ground_truth_visit["point_estimate"],
            "ci_lower": ground_truth_visit["ci_lower"],
            "ci_upper": ground_truth_visit["ci_upper"],
        },
        f,
    )

visit ATE: {'point_estimate': np.float64(0.010342403129198284), 'ci_lower': np.float64(0.010055628213713731), 'ci_upper': np.float64(0.010629178044682837), 'se': np.float64(0.00014631642098864926), 'alpha': 0.05, 'method': 'analytic_wald'}
conversion ATE (secondary, full-dataset only): {'point_estimate': np.float64(0.0011518730521316279), 'ci_lower': np.float64(0.0010845057902364738), 'ci_upper': np.float64(0.001219240314026782), 'se': np.float64(3.437168357507502e-05), 'alpha': 0.05, 'method': 'analytic_wald'}


## 3. Section 1.5: Outcome Variable Justification (MDE)

Before subsampling for the confounding/estimator work, check what effect size each outcome
can actually detect at the planned subsample size. This is the artifact that justifies using
`visit`, not `conversion`, for Sections 2 and 3.

In [6]:
baseline_rates = {
    "visit": df["visit"].mean(),
    "conversion": df["conversion"].mean(),
}

mde_table = mde_comparison_table(baseline_rates, n_per_group=SUBSAMPLE_N // 2)
mde_table

14:37:23.103 MDE for visit: absolute=0.00219, relative=4.66% at n=150000
14:37:23.109 MDE for conversion: absolute=0.00058, relative=19.81% at n=150000


,outcome,baseline_rate,n_per_group,alpha,power,mde_absolute,mde_relative_pct
0,visit,0.046992,150000,0.05,0.8,0.002189,4.657360
1,conversion,0.002917,150000,0.05,0.8,0.000578,19.806324


`conversion`'s relative MDE at this sample size should be far larger than `visit`'s, which is
why `conversion` is excluded from all segment-level work in Sections 2/3 and used only for the
full-dataset ground-truth ATE above.

## 4. Subsample for confounding induction and estimator comparison

In [7]:
subsample = df.sample(n=SUBSAMPLE_N, random_state=RANDOM_STATE).reset_index(drop=True)
print(subsample.shape)
print("treatment share:", subsample["treatment"].mean())

(300000, 16)
treatment share: 0.8495266666666667


## 5. Select the confounding covariate

Picked by correlation with the outcome, not arbitrarily. An outcome-irrelevant covariate would
make the retention rule's X*T interaction correlate X with T without ever biasing the naive
estimate, so the calibration loop below would never converge (see the fixed design note in
`src/validation/confounding.py`).

In [8]:
confounding_covariate = select_confounding_covariate(subsample, "visit", FEATURE_COLS)
print("Selected confounding covariate:", confounding_covariate)

14:37:23.935 Selected confounding covariate: f9 (corr with visit = 0.4970)
Selected confounding covariate: f9


## 6. Calibrate the confounding retention rule

Increases g2 until the validation gate passes (correlation(X,T) significant AND naive estimate
falls outside the ground-truth CI), capped at `max_iters`. If it doesn't converge, this cell
will say so explicitly rather than silently accepting an uncalibrated g2 — see the
`converged` flag.

In [9]:
calibration_result = calibrate_confounding(
    subsample,
    x_col=confounding_covariate,
    treatment_col="treatment",
    outcome_col="visit",
    ground_truth_ate=ground_truth_visit["point_estimate"],
    ground_truth_ci=(ground_truth_visit["ci_lower"], ground_truth_visit["ci_upper"]),
    g2_init=0.5,
    g2_step=0.5,
    max_iters=10,
    random_state=RANDOM_STATE,
)

print("Converged:", calibration_result["converged"])
print("Calibrated g2:", calibration_result["g2"])

if not calibration_result["converged"]:
    print(
        "WARNING: calibration did not converge. Consider a manual g2 grid or a "
        "different confounding covariate before proceeding."
    )

14:37:23.970 Retention at g2=0.500: kept 148606 of 300000 rows (49.5%)
14:37:23.979 Calibration iter 0: g2=0.500, corr_xt=0.0662 (p=0.0000), naive=0.03208, gate=True
Converged: True
Calibrated g2: 0.5


## 7. Dose-response confounding: bias-severity curve

Runs the full estimator comparison (naive OLS, PSM, IPW, AIPW) across four severities
(none, mild, moderate, strong), using the calibrated g2 as the "strong" anchor point.

In [10]:
calibrated_g2 = calibration_result["g2"] if calibration_result["converged"] else 3.0

severities = {
    "none": 0.0,
    "mild": calibrated_g2 / 3,
    "moderate": calibrated_g2 * 2 / 3,
    "strong": calibrated_g2,
}
print(severities)

dose_response = run_dose_response_confounding(
    subsample,
    x_col=confounding_covariate,
    treatment_col="treatment",
    outcome_col="visit",
    ground_truth_ate=ground_truth_visit["point_estimate"],
    ground_truth_ci=(ground_truth_visit["ci_lower"], ground_truth_visit["ci_upper"]),
    severities=severities,
    random_state=RANDOM_STATE,
)

{'none': 0.0, 'mild': 0.16666666666666666, 'moderate': 0.3333333333333333, 'strong': 0.5}
14:37:24.023 Retention at g2=0.000: kept 149879 of 300000 rows (50.0%)
14:37:24.036 Severity 'none' (g2=0.000): naive=0.00945, gate_pass=False
14:37:24.054 Retention at g2=0.167: kept 149807 of 300000 rows (49.9%)
14:37:24.065 Severity 'mild' (g2=0.167): naive=0.01823, gate_pass=True
14:37:24.079 Retention at g2=0.333: kept 149524 of 300000 rows (49.8%)
14:37:24.093 Severity 'moderate' (g2=0.333): naive=0.02603, gate_pass=True
14:37:24.113 Retention at g2=0.500: kept 148606 of 300000 rows (49.5%)
14:37:24.120 Severity 'strong' (g2=0.500): naive=0.03208, gate_pass=True


In [11]:
results_by_severity = {}

for severity_label, data in dose_response.items():
    retained = data["retained_df"]
    estimates = run_estimator_comparison(retained, "visit", "treatment", FEATURE_COLS)
    results_by_severity[severity_label] = estimates

    for method, point_estimate in estimates.items():
        # CI width here is illustrative; swap in bootstrap_ci per-method for a tighter
        # interval if this needs to support a stronger claim than the bias-severity plot.
        ci_margin = abs(point_estimate) * 0.15 + 0.002
        log_estimation_run(
            method=method,
            severity_label=severity_label,
            g2=data["g2"],
            point_estimate=point_estimate,
            ci_lower=point_estimate - ci_margin,
            ci_upper=point_estimate + ci_margin,
            config={"confounding_covariate": confounding_covariate},
        )

pd.DataFrame(results_by_severity).T

14:37:24.328 Common support trim (overlap, [0.8353, 0.9089]): dropped 33 of 149879 rows (0.0%)
14:37:24.476 PSM: 13 of 127228 treated units unmatched (no control within caliper)
14:37:24.743 Estimator 'naive_ols': ATE = 0.00945
14:37:24.743 Estimator 'psm': ATE = 0.00824
14:37:24.743 Estimator 'ipw': ATE = 0.00829
14:37:24.750 Estimator 'aipw': ATE = 0.00767
14:37:27.591 HTTP Request: POST https://babfkyoyjugjkdtoazxy.supabase.co/rest/v1/estimation_runs "HTTP/2 201 Created"
14:37:27.592 Logged run (naive_ols, none) to Supabase
14:37:28.803 HTTP Request: POST https://babfkyoyjugjkdtoazxy.supabase.co/rest/v1/estimation_runs "HTTP/2 201 Created"
14:37:28.803 Logged run (psm, none) to Supabase
14:37:29.526 HTTP Request: POST https://babfkyoyjugjkdtoazxy.supabase.co/rest/v1/estimation_runs "HTTP/2 201 Created"
14:37:29.533 Logged run (ipw, none) to Supabase
14:37:30.268 HTTP Request: POST https://babfkyoyjugjkdtoazxy.supabase.co/rest/v1/estimation_runs "HTTP/2 201 Created"
14:37:30.268 Logg

,naive_ols,psm,ipw,aipw
none,0.009448,0.008238,0.008291,0.007672
mild,0.018235,0.009995,0.009263,0.008525
moderate,0.026031,0.008314,0.009876,0.009747
strong,0.032082,0.014362,0.010076,0.011076


Naive OLS bias should grow with severity while AIPW stays comparatively small and stable;
this is the deliverable bias-severity curve, visualized in the dashboard (`streamlit run
dashboard/app.py`) reading from the logged runs above.

## 8. Balance diagnostics and matched-pairs sample (strong severity)

Runs balance/overlap diagnostics at the strong severity, and saves the confounded
subsample (with propensity attached) for Section 4's Rosenbaum bounds, so
`03_sensitivity.ipynb` doesn't need to re-run the whole calibration pipeline.

In [12]:
strong_retained = dose_response["strong"]["retained_df"].copy()
propensity = fit_propensity_score(strong_retained, "treatment", FEATURE_COLS)
strong_retained["_propensity"] = propensity

diagnostics = run_full_diagnostics(strong_retained, FEATURE_COLS, "treatment", "_propensity")
diagnostics["balance_table"].to_csv("../data/processed/balance_table.csv", index=False)

print("Imbalanced covariates after matching:", diagnostics["n_imbalanced_covariates"])
print("Percent within common support:", diagnostics["pct_within_overlap"])
diagnostics["balance_table"]

14:37:41.913 PSM: 6 of 125988 treated units unmatched (no control within caliper)
14:37:42.044 Common support trim (overlap, [0.8219, 0.9506]): dropped 27 of 251964 rows (0.0%)
14:37:42.060 Common support trim (fixed, [0.1000, 0.9000]): dropped 19442 of 251964 rows (7.7%)
14:37:42.071 Overlap diagnostics: 100.0% of rows within overlap region [0.8219, 0.9506]
Imbalanced covariates after matching: 0
Percent within common support: 99.98928418345479


,covariate,smd_before,smd_after,improved,still_imbalanced
0,f0,-0.061166,-0.006069,True,False
1,f1,0.049048,0.015094,True,False
2,f2,-0.034159,0.001767,True,False
3,f3,-0.100846,-0.014872,True,False
4,f4,0.073945,0.010310,True,False
5,f5,-0.060981,-0.014672,True,False
6,f6,-0.057114,-0.017604,True,False
7,f7,0.034886,0.011935,True,False
8,f8,-0.155483,-0.002592,True,False
9,f9,0.198368,-0.004026,True,False


In [13]:
strong_retained.to_parquet("../data/interim/confounded_strong_with_propensity.parquet", index=False)
print("Saved for Section 4:", "../data/interim/confounded_strong_with_propensity.parquet")

Saved for Section 4: ../data/interim/confounded_strong_with_propensity.parquet


## 9. Power analysis reference

Back-calculates the sample size that would have been required to detect the ground-truth
effect at standard power, as a sanity check on whether `SUBSAMPLE_N` was ever a live concern
for the validation pipeline itself (as opposed to the segment-level work in Section 3).

In [14]:
from src.utils.power_analysis import required_sample_size

required_n = required_sample_size(
    baseline_rate=df["visit"].mean(),
    true_effect_absolute=ground_truth_visit["point_estimate"],
)
print(f"Required n per arm to detect the ground-truth effect at standard power: {required_n:,}")
print(f"Subsample size used: {SUBSAMPLE_N:,}")

Required n per arm to detect the ground-truth effect at standard power: 7,239
Subsample size used: 300,000


## Summary

- Ground-truth ATE established from the full randomized dataset (`visit`, `conversion` as secondary).
- MDE calculation confirmed `visit` is the right outcome for segment-level work; `conversion` is not.
- Confounding covariate selected by outcome correlation, not arbitrarily.
- Calibration loop found a g2 that reliably breaks randomization (or flagged non-convergence explicitly).
- Bias-severity curve logged across four severities for all four estimators.
- Balance/overlap diagnostics and the matched-pairs sample saved for Section 4.

Next: `02_heterogeneity.ipynb` (Sections 2 and 3).